# Modul B · Kapitel 4 — Das richtige Modell auswählen

> 🛠️ **Workshop-Version:** Bearbeite die zwei markierten Aufgaben.

**Lernziel:** Du kannst Modelle an der eigenen Aufgabe statt an Leaderboards vergleichen.

Dieses Notebook folgt einem kurzen Pfad: Begriff verstehen → Rechnung oder
Messung durchführen → Ergebnis für eine Deployment-Entscheidung nutzen.
Programmiert werden nur zwei Kernstellen: eine belastbare Messung und eine Auswahlregel. Hilfs- und
Visualisierungscode ist bewusst vorgegeben.


## 0 · Setup

Die nächsten Zellen laden Eval-Fälle, Kandidaten und Model-Card-Daten. Live-Vergleiche benötigen die konfigurierten Modelle.


In [ ]:
import sys
from pathlib import Path

# helfer.py liegt neben dem Notebook. Der Suchlauf findet es auch, wenn das
# Arbeitsverzeichnis woanders liegt — etwa in Colab.
for kandidat in [Path.cwd(), Path.cwd() / "04_deployment", *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

try:
    import matplotlib
    import pandas
except ImportError:
    %pip install -q matplotlib pandas openai tiktoken
    import matplotlib

import json
import random
import re
import statistics
import textwrap
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

import helfer
from helfer import GB, lade_daten, messe_anfrage, zeige_tabelle

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"Standardmodell: {helfer.MODELL} über {helfer.BASIS_URL}")
print(f"reasoning_effort: {helfer.REASONING}")
print("Setup fertig ✔")

In [ ]:
# ▶️ Die Eval-Menge: 30 CVE-Kurzbeschreibungen mit Sollwerten
EVALSET = lade_daten("05_evalset")
FAELLE = EVALSET["faelle"]
SEVERITIES = EVALSET["wertelisten"]["severity"]
AKTIONEN = EVALSET["wertelisten"]["action"]
FELDER = ["severity", "component", "action"]

print(f"{len(FAELLE)} Fälle   (Stand {EVALSET['stand']})")
print(f"severity: {', '.join(SEVERITIES)}")
print(f"action:   {', '.join(AKTIONEN)}")
print()
print("  Verteilung severity:", dict(Counter(f["severity"] for f in FAELLE).most_common()))
print("  Verteilung action:  ", dict(Counter(f["action"] for f in FAELLE).most_common()))
print()

beispiel = FAELLE[0]
print(f"[{beispiel['id']}] {textwrap.fill(beispiel['text'], 96, subsequent_indent='      ')}")
print(f"      Soll: severity={beispiel['severity']!r}  component={beispiel['component']!r}  "
      f"action={beispiel['action']!r}")

In [ ]:
# ▶️ Die drei Kandidaten und ihre Model Cards
KANDIDATEN = lade_daten("05_kandidaten")["kandidaten"]
MODELL_NACH_NAME = {m["name"]: m for m in lade_daten("model_cards")["modelle"]}


def ollama_groesse_gb(modell):
    """Größe eines Modells auf der Platte in GB, gemeldet vom lokalen Ollama-Dienst."""
    adresse = helfer.BASIS_URL.replace("/v1", "") + "/api/tags"
    try:
        with urllib.request.urlopen(adresse, timeout=5) as antwort:
            eintraege = json.load(antwort)["models"]
    except Exception as fehler:
        print(f"Ollama nicht erreichbar ({type(fehler).__name__}) — Größe unbekannt.")
        return None

    for eintrag in eintraege:
        if eintrag["name"].split(":")[0] == modell.split(":")[0]:
            return eintrag["size"] / GB
    return None


for k in KANDIDATEN:
    k["speicher_gb"] = ollama_groesse_gb(k["ollama"])
    k["karte_daten"] = MODELL_NACH_NAME[k["karte"]]

zeige_tabelle([{
    "Ollama": k["ollama"],
    "Parameter": f"{k['parameter'] / 1e9:.2f} Mrd.",
    "Quantisierung": k["quantisierung"],
    "auf der Platte (GB)": k["speicher_gb"],
    "Reasoning": "ja" if k["reasoning"] else "nein",
    "Model Card": k["karte"] + ("" if k["karte_genau"] else "  (Stellvertreter)"),
    "Lizenz": k["karte_daten"]["lizenz"],
} for k in KANDIDATEN])

print("Die Lizenz kommt aus model_cards.json. Wo die Card ein Stellvertreter ist, gibt es")
print("für die lokale Ollama-Variante keinen eigenen Eintrag — die Lizenz gilt für die")
print("Familie, die Parameterzahl steht in der Spalte davor.")

## 1 · Drei Quellen, drei Fragen

Die Model Card klärt Eigenschaften und Lizenz, ein Leaderboard liefert Fremdmessungen, die eigene Eval beantwortet die entscheidende Frage: Funktioniert das Modell für unsere Aufgabe?


In [ ]:
# ▶️ Die drei Quellen als Prüfliste — jede beantwortet eine Frage
QUELLEN = [
    {"Quelle": "Hugging Face", "beantwortet": "Was ist das Modell und darf ich es benutzen?",
     "Einheit": "Parameter, Lizenz, Downloads",
     "misst nicht": "Qualität an einer bestimmten Aufgabe"},
    {"Quelle": "LMArena", "beantwortet": "Welche Antwort gefällt Menschen besser?",
     "Einheit": "Elo aus paarweisen Vergleichen",
     "misst nicht": "Format­treue, Extraktion, dein Fachgebiet"},
    {"Quelle": "artificialanalysis.ai", "beantwortet": "Was kostet eine Anfrage bei wem?",
     "Einheit": "Tokens/s, Sekunden, Preis je 1M Tokens",
     "misst nicht": "Qualität, lokale Hardware, eigene Modelle"},
    {"Quelle": "deine Eval-Menge", "beantwortet": "Löst das Modell meine Aufgabe?",
     "Einheit": "Trefferquote je Feld, Sekunden je Fall",
     "misst nicht": "alles, was nicht in der Menge steht"},
]

zeige_tabelle(QUELLEN)
print("Die vierte Zeile ist die einzige, die niemand außer dir erstellen kann.")

## 2 · Eine eigene Eval-Menge

Die Fälle müssen repräsentativ sein und Sollwerte besitzen. Entwicklung und Prüfung bleiben getrennt, damit Prompt-Tuning das Endergebnis nicht schönfärbt.


In [ ]:
# Vorgegebene Hilfsfunktion
def baue_evalset(faelle, anteil_pruefung=0.6, saat=7):
    """Teilt die Fälle geschichtet nach severity in Entwicklungs- und Prüfmenge."""
    entwicklung, pruefung = [], []

    for stufe in SEVERITIES:
        # Erst sortieren, dann mischen: sonst hängt das Ergebnis an der
        # Reihenfolge in der Datei und nicht nur an der Saat.
        gruppe = sorted((f for f in faelle if f["severity"] == stufe),
                        key=lambda f: f["id"])
        random.Random(saat).shuffle(gruppe)

        grenze = round(len(gruppe) * anteil_pruefung)
        pruefung += gruppe[:grenze]
        entwicklung += gruppe[grenze:]

    return {"entwicklung": sorted(entwicklung, key=lambda f: f["id"]),
            "pruefung": sorted(pruefung, key=lambda f: f["id"])}

In [ ]:
# ▶️ Entwicklungs- und Prüfmenge einmalig festlegen
geteilt = baue_evalset(FAELLE)
ENTWICKLUNG = geteilt["entwicklung"]
PRUEFUNG = geteilt["pruefung"]

print(f"Entwicklung: {len(ENTWICKLUNG)} Fälle")
print(f"Prüfung:     {len(PRUEFUNG)} Fälle")


## 3 · Fair messen

Alle Modelle erhalten denselben Prompt, dieselben Fälle und dieselben Decoding-Einstellungen. Wir messen Format, Feldtreue und Laufzeit.


In [ ]:
# ▶️ Der Prompt und der Parser — fertig, du brauchst beides in Aufgabe 1
AUFGABE_KURZ = (
    "Extract the severity, the affected component and the recommended action from the "
    "vulnerability description below.\n"
    "Answer with a single JSON object with the keys severity, component and action, "
    "and nothing else."
)

AUFGABE = (
    "Extract three fields from the vulnerability description below.\n"
    "\n"
    "severity — from the CVSS score in the text:\n"
    "  9.0-10.0 critical, 7.0-8.9 high, 4.0-6.9 medium, 0.1-3.9 low.\n"
    "component — the affected product, without version numbers and without extra words.\n"
    "action — the single most urgent step, exactly one of:\n"
    "  upgrade             a fixed release or version of the product is available\n"
    "  patch               the vendor published a patch or hotfix for the installed version\n"
    "  rotate_credentials  secrets were exposed and have to be replaced\n"
    "  disable_feature     no fix exists, the affected function has to be turned off\n"
    "  isolate             the system has to be taken off the network until it is fixed\n"
    "  monitor             no fix and no workaround, only the logs can be watched\n"
    "\n"
    "Answer with a single JSON object with the keys severity, component and action, "
    "and nothing else."
)

ZAUN = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$")


def baue_prompt(aufgabe, text):
    """Anweisung und CVE-Beschreibung zu einem Prompt zusammensetzen."""
    return f"{aufgabe}\n\n{text}"


def lies_json(antwort):
    """Die Antwort als Dictionary — oder None, wenn sie unbrauchbar ist."""
    roh = ZAUN.sub("", (antwort or "").strip()).strip()
    try:
        daten = json.loads(roh)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(daten, dict) or not all(feld in daten for feld in FELDER):
        return None
    return daten


def passt(vorhersage, soll):
    """Vergleicht zwei Feldwerte ohne Rücksicht auf Groß- und Kleinschreibung."""
    if vorhersage is None:
        return False
    return str(vorhersage).strip().lower() == str(soll).strip().lower()


print(AUFGABE)
print()
print(f"Prompt-Länge: {helfer.zaehle_tokens(baue_prompt(AUFGABE, FAELLE[0]['text']))} Tokens "
      f"(kurze Fassung: {helfer.zaehle_tokens(baue_prompt(AUFGABE_KURZ, FAELLE[0]['text']))})")

In [ ]:
# ▶️ Fünf Antworten, wie Modelle sie liefern — fest kodiert für den Selbsttest
PROBE_FAELLE = [
    {"id": "P1", "severity": "critical", "component": "Devolutions Server", "action": "upgrade"},
    {"id": "P2", "severity": "high", "component": "nginx", "action": "upgrade"},
    {"id": "P3", "severity": "high", "component": "Jenkins", "action": "rotate_credentials"},
    {"id": "P4", "severity": "medium", "component": "GitLab CE", "action": "upgrade"},
    {"id": "P5", "severity": "high", "component": "Jenkins", "action": "rotate_credentials"},
]

PROBE_ANTWORTEN = [
    '{"severity": "critical", "component": "Devolutions Server", "action": "upgrade"}',
    '```json\n{"severity": "high", "component": "nginx", "action": "upgrade"}\n```',
    "The severity is high. The affected component is Jenkins, and the exposed credentials "
    "have to be rotated.",
    '{"level": "medium", "system": "GitLab CE", "step": "upgrade"}',
    '{"severity": "high", "component": "jenkins", "action": "monitor"}',
]

for fall, antwort in zip(PROBE_FAELLE, PROBE_ANTWORTEN):
    gelesen = lies_json(antwort)
    print(f"{fall['id']}  {'lesbar ' if gelesen else 'unlesbar'}  {antwort[:62]!r}")

### 🛠️ Aufgabe 1 — Ein Modell fair bewerten

Vergleiche rohe Antworten mit den Sollwerten und berechne für alle Modelle dieselben Qualitätsquoten.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def zaehle_quoten(antworten, faelle):
    """Vergleicht rohe Antworttexte mit den Sollwerten und gibt fünf Quoten zurück."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: zaehle_quoten() implementieren")


def bewerte_modell(modell, faelle, aufgabe=AUFGABE):
    """Schickt jeden Fall an das Modell und gibt Qualität und Kosten zurück."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: bewerte_modell() implementieren")


In [ ]:
# ✅ Selbsttest — erst auf den festen Antworten, dann ein kurzer Lauf am Modell
q = zaehle_quoten(PROBE_ANTWORTEN, PROBE_FAELLE)

assert set(q) == {"json_quote", "severity", "component", "action", "gesamt"}, \
    f"Erwartet werden fünf Schlüssel, geliefert wurden: {sorted(q)}"
assert round(q["json_quote"], 2) == 0.60, \
    f"Drei der fünf Antworten sind lesbar (der Zaun zählt nicht), gemeldet: {q['json_quote']:.2f}"
assert round(q["severity"], 2) == 0.60, f"Drei severity-Treffer, gemeldet: {q['severity']:.2f}"
assert round(q["component"], 2) == 0.60, \
    f"'jenkins' und 'Jenkins' sind derselbe Treffer, gemeldet: {q['component']:.2f}"
assert round(q["action"], 2) == 0.40, f"Zwei action-Treffer, gemeldet: {q['action']:.2f}"
assert round(q["gesamt"], 2) == 0.40, \
    f"Nur P1 und P2 stimmen in allen drei Feldern, gemeldet: {q['gesamt']:.2f}"

probe = bewerte_modell(helfer.MODELL, ENTWICKLUNG[:3])
assert probe["faelle"] == 3 and probe["modell"] == helfer.MODELL
assert len(probe["antworten"]) == 3, "Für jeden Fall gehört eine Antwort in das Ergebnis"
assert probe["sekunden_je_fall"] > 0 and probe["tokens_je_fall"] > 0, "Es wurde nichts gemessen"
assert probe["ttft_je_fall"] > 0 and probe["tokens_pro_sekunde"] > 0, "TTFT und Rate fehlen"
assert probe["sekunden_gesamt"] >= probe["sekunden_je_fall"], "Die Summe ist kein Median"

print("✅ Aufgabe 1 gelöst")
print()
for schluessel, wert in q.items():
    print(f"  {schluessel:<12} {wert:>5.0%}")
print()
print(f"Kurzer Lauf mit {helfer.MODELL} über 3 Fälle: "
      f"{probe['gesamt']:.0%} gesamt, {probe['sekunden_je_fall']:.2f} s je Fall "
      f"(davon {probe['ttft_je_fall']:.2f} s bis zum ersten Token), "
      f"{probe['tokens_je_fall']:.0f} Tokens je Fall")

In [ ]:
# ▶️ Prompt-Entwicklung auf der Entwicklungsmenge — nicht auf der Prüfmenge
def ausserhalb_der_liste(antworten):
    """Zählt Antworten, deren action gar nicht in der Werteliste steht."""
    gelesen = [lies_json(a) for a in antworten]
    return sum(str((g or {}).get("action")).strip().lower() not in AKTIONEN for g in gelesen)


ENTWICKLUNGSLAEUFE = {}
for name, aufgabe in [("kurze Anweisung", AUFGABE_KURZ), ("mit Wertelisten", AUFGABE)]:
    ENTWICKLUNGSLAEUFE[name] = bewerte_modell(helfer.MODELL, ENTWICKLUNG, aufgabe)

zeige_tabelle([{
    "Anweisung": name,
    "JSON": e["json_quote"],
    "severity": e["severity"],
    "component": e["component"],
    "action": e["action"],
    "gesamt": e["gesamt"],
    "action außerhalb der Liste": f"{ausserhalb_der_liste(e['antworten'])} von {e['faelle']}",
    "s/Fall": e["sekunden_je_fall"],
} for name, e in ENTWICKLUNGSLAEUFE.items()])

print(f"{helfer.MODELL} auf {len(ENTWICKLUNG)} Fällen der Entwicklungsmenge.")
print("Quoten als Anteil von 1,0.")
print()
for name, e in ENTWICKLUNGSLAEUFE.items():
    werte = Counter(str((lies_json(a) or {}).get("action")).lower() for a in e["antworten"])
    print(f"{name:<18} gelieferte action-Werte: "
          + ", ".join(f"{w} ({n}×)" for w, n in werte.most_common()))

## 4 · Kandidaten vergleichen

Ein größeres Modell ist nicht automatisch die beste Wahl. Entscheidend ist, welches Modell die Mindestqualität mit vertretbarer Zeit und Kosten erreicht.


In [ ]:
# ▶️ Die Messreihe: drei Modelle über die Prüfmenge
ERGEBNISSE = []
for k in KANDIDATEN:
    ergebnis = bewerte_modell(k["ollama"], PRUEFUNG)
    ergebnis["anzeige"] = k["anzeige"]
    ERGEBNISSE.append(ergebnis)
    print(f"{k['ollama']:<14} {ergebnis['gesamt']:>4.0%} gesamt   "
          f"{ergebnis['sekunden_je_fall']:>5.2f} s/Fall   "
          f"{ergebnis['sekunden_gesamt']:>6.1f} s für {ergebnis['faelle']} Fälle")

print()
print("Qualität")
zeige_tabelle([{
    "Modell": e["anzeige"],
    "JSON lesbar": e["json_quote"],
    "severity": e["severity"],
    "component": e["component"],
    "action": e["action"],
    "alle drei": e["gesamt"],
} for e in ERGEBNISSE])

print("Kosten")
zeige_tabelle([{
    "Modell": e["anzeige"],
    "s/Fall": e["sekunden_je_fall"],
    "davon TTFT (s)": e["ttft_je_fall"],
    "Tokens/s": e["tokens_pro_sekunde"],
    "Tokens/Fall": e["tokens_je_fall"],
    "Sekunden gesamt": e["sekunden_gesamt"],
} for e in ERGEBNISSE])

print(f"{len(PRUEFUNG)} Fälle der Prüfmenge je Modell, dieselbe Anweisung, temperature 0.")
print("Alle Zeitangaben sind Mediane über die Fälle.")

In [ ]:
# ▶️ Qualität gegen Zeit — und die Trefferquote je Feld
fig, (links, rechts) = plt.subplots(1, 2, figsize=(13, 4.8))

for ergebnis, farbe in zip(ERGEBNISSE, [TEAL, BLAU, ORANGE]):
    links.scatter(ergebnis["sekunden_je_fall"], ergebnis["gesamt"] * 100,
                  s=260, color=farbe, zorder=3, edgecolor="white", linewidth=1.5)
    links.annotate(f"{ergebnis['anzeige']}\n{ergebnis['gesamt']:.0%} · "
                   f"{ergebnis['sekunden_je_fall']:.2f} s",
                   (ergebnis["sekunden_je_fall"], ergebnis["gesamt"] * 100),
                   textcoords="offset points", xytext=(0, 20), ha="center",
                   fontsize=9, color=farbe, fontweight="bold")

links.set_xlabel("Sekunden je Fall (Median)")
links.set_ylabel("alle drei Felder richtig (%)")
links.set_xlim(0, max(e["sekunden_je_fall"] for e in ERGEBNISSE) * 1.45)
links.set_ylim(-4, max(e["gesamt"] for e in ERGEBNISSE) * 100 + 26)
links.set_title("Die Modellwahl ist eine Kurve, kein Sieger")

breite = 0.26
stellen = np.arange(len(FELDER) + 1)
for i, (ergebnis, farbe) in enumerate(zip(ERGEBNISSE, [TEAL, BLAU, ORANGE])):
    werte = [ergebnis[f] * 100 for f in FELDER] + [ergebnis["gesamt"] * 100]
    balken = rechts.bar(stellen + (i - 1) * breite, werte, breite,
                        label=ergebnis["anzeige"], color=farbe)
    rechts.bar_label(balken, fmt="%.0f", fontsize=8, padding=2)

rechts.set_xticks(stellen)
rechts.set_xticklabels(FELDER + ["alle drei"])
rechts.set_ylabel("Trefferquote (%)")
rechts.set_ylim(0, 108)
rechts.legend(frameon=False, fontsize=9)
rechts.set_title("Wo die Modelle sich unterscheiden")

plt.tight_layout()
plt.show()

In [ ]:
# ▶️ Woran die Modelle scheitern: die häufigsten Verwechslungen im Feld action
for ergebnis in ERGEBNISSE:
    gelesen = [lies_json(a) for a in ergebnis["antworten"]]
    verwechslungen = Counter(
        (f["action"], str((g or {}).get("action")).lower())
        for g, f in zip(gelesen, PRUEFUNG)
        if not passt((g or {}).get("action"), f["action"])
    )
    print(f"{ergebnis['anzeige']}  —  {len(list(verwechslungen.elements()))} von "
          f"{len(PRUEFUNG)} Fällen falsch")
    for (soll, ist), anzahl in verwechslungen.most_common(4):
        print(f"    soll {soll:<19} geliefert {ist:<19} {anzahl}×")
    print()

## 5 · Aus Messwerten entscheiden

Zuerst gelten harte Anforderungen wie Mindestqualität und Speicher. Unter den verbleibenden Kandidaten gewinnt die günstigste oder schnellste Option.


### 🛠️ Aufgabe 2 — Eine Auswahlregel anwenden

Wähle das schnellste Modell oberhalb der Mindestqualität. Erreicht keines die Schwelle, gib das qualitativ beste zurück.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def waehle_modell(ergebnisse, mindestqualitaet):
    """Das schnellste Modell, das die Mindestqualität erreicht — sonst das beste."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 2: waehle_modell() implementieren")


In [ ]:
# ✅ Selbsttest — auf erfundenen Ergebnissen, damit das Urteil feststeht
PROBE_ERGEBNISSE = [
    {"modell": "winzig", "gesamt": 0.10, "sekunden_je_fall": 0.40},
    {"modell": "mittel", "gesamt": 0.50, "sekunden_je_fall": 1.00},
    {"modell": "gross", "gesamt": 0.62, "sekunden_je_fall": 8.00},
]

a = waehle_modell(PROBE_ERGEBNISSE, 0.05)
assert a["modell"] == "winzig" and a["erfuellt"], "Alle erfüllen 5 % — dann zählt die Zeit"
assert round(a["zeitfaktor"], 2) == 1.00, "Der schnellste Kandidat hat den Zeitfaktor 1"

b = waehle_modell(PROBE_ERGEBNISSE, 0.45)
assert b["modell"] == "mittel", f"Bei 45 % fällt 'winzig' heraus, gewählt wurde {b['modell']!r}"
assert round(b["zeitfaktor"], 1) == 2.5, f"2,5-fache Zeit gegenüber 'winzig': {b['zeitfaktor']}"

c = waehle_modell(PROBE_ERGEBNISSE, 0.60)
assert c["modell"] == "gross" and round(c["zeitfaktor"], 0) == 20, \
    "Bei 60 % bleibt nur 'gross' — zum zwanzigfachen Zeitaufwand"

d = waehle_modell(PROBE_ERGEBNISSE, 0.90)
assert not d["erfuellt"], "90 % erreicht keiner — das muss im Ergebnis stehen"
assert d["modell"] == "gross", "Wenn keiner reicht, steht das beste Ergebnis daneben"

print("✅ Aufgabe 2 gelöst")
print()
for schwelle in (0.10, 0.25, 0.40, 0.60, 0.80):
    w = waehle_modell(ERGEBNISSE, schwelle)
    zeichen = "✓" if w["erfuellt"] else "✗"
    print(f"{zeichen} Mindestqualität {schwelle:>4.0%}  →  {w['modell']:<14} "
          f"{w['qualitaet']:>4.0%}, {w['sekunden_je_fall']:>5.2f} s/Fall, "
          f"Zeitfaktor {w['zeitfaktor']:>5.1f}")
    print(f"     {w['grund']}")

## 6 · Unsicherheit sichtbar machen

Kleine Eval-Mengen und nichtdeterministische Ausgaben erzeugen Streuung. Ergebnisse sind Schätzungen, keine Naturkonstanten.


In [ ]:
# Vorgegebene Hilfsfunktion
def wiederhole_messung(modell, faelle, laeufe=2):
    """Misst dasselbe Modell mehrfach auf derselben Menge und gibt die Streuung zurück."""
    laufergebnisse = [bewerte_modell(modell, faelle) for _ in range(laeufe)]

    qualitaet = [e["gesamt"] for e in laufergebnisse]
    zeiten = [e["sekunden_je_fall"] for e in laufergebnisse]

    return {
        "modell": modell,
        "laeufe": laeufe,
        "qualitaet": qualitaet,
        "sekunden_je_fall": zeiten,
        "spanne_qualitaet": max(qualitaet) - min(qualitaet),
        "spanne_zeit": max(zeiten) - min(zeiten),
        # Ein einziger Fall verschiebt die Quote um 1/n. Feiner als das kann
        # keine Messung an dieser Menge sein.
        "fehlerschranke": 1 / len(faelle),
    }

## 7 · Die Auswahlmatrix

Qualität und Zeit werden gemessen; Lizenz, Speicher und Betriebsort kommen als harte Anforderungen hinzu.


In [ ]:
# ▶️ Die Zeilen der Matrix — Messwerte, Model Card und Betriebsangaben zusammen
def gewichte_fp16_gb(karte):
    """Was das Modell in voller Präzision belegen würde, laut Model Card."""
    return karte["parameter_gesamt"] * 16 / 8 / GB


AUSWAHL = []
for kandidat, ergebnis in zip(KANDIDATEN, ERGEBNISSE):
    karte = kandidat["karte_daten"]
    AUSWAHL.append({
        "kandidat": kandidat["anzeige"],
        "ollama": kandidat["ollama"],
        "qualitaet": ergebnis["gesamt"],
        "sekunden_je_fall": ergebnis["sekunden_je_fall"],
        "speicher_gb": kandidat["speicher_gb"],
        "fp16_gb": gewichte_fp16_gb(karte),
        "lizenz": karte["lizenz"],
        "betrieb": kandidat["betrieb"],
    })

zeige_tabelle([{
    "Modell": z["kandidat"],
    "Qualität": z["qualitaet"],
    "s/Fall": z["sekunden_je_fall"],
    "quantisiert (GB)": z["speicher_gb"],
    "FP16 laut Card (GB)": z["fp16_gb"],
    "Lizenz": z["lizenz"],
    "Betrieb": z["betrieb"],
} for z in AUSWAHL])

print("Die Spalte „quantisiert“ ist die Datei, die Ollama vorhält — das ist der Speicher, den")
print("das Modell wirklich braucht. Die Spalte daneben ist die Rechnung aus der Model Card für")
print("volle Präzision; sie zeigt, was die Quantisierung einspart.")

In [ ]:
# ▶️ Zwei Situationen, dieselben drei Modelle
SITUATIONEN = [
    {
        "titel": "Prototyp auf dem Laptop",
        "text": "Eine Person probiert die Extraktion auf ihrem Arbeitsgerät aus, 16 GB "
                "Arbeitsspeicher, davon höchstens die Hälfte für das Modell. Sie wartet bei "
                "jedem Fall zu und liest die Ergebnisse nach. Es geht darum, ob die Aufgabe "
                "überhaupt trägt.",
        "mindestqualitaet": 0.10,
        "zeitbudget_s": 3.0,
        "speicher_gb": 8.0,
        "lizenzen": None,
    },
    {
        "titel": "Interner Dienst für fünfzig Personen",
        "text": "Der Dienst läuft auf einer Karte mit 24 GB im eigenen Rechenzentrum und "
                "schreibt die Felder ohne Nachkontrolle in ein Ticketsystem. Falsche Felder "
                "verursachen Arbeit an anderer Stelle, deshalb liegt die Anforderung hoch. "
                "Die Antwort darf einen Moment brauchen.",
        "mindestqualitaet": 0.40,
        "zeitbudget_s": 6.0,
        "speicher_gb": 24.0,
        "lizenzen": None,
    },
]

for s in SITUATIONEN:
    print(f"{s['titel']}")
    print(textwrap.fill(s["text"], 96, initial_indent="   ", subsequent_indent="   "))
    print(f"   Anforderungen: Qualität ≥ {s['mindestqualitaet']:.0%}   "
          f"≤ {s['zeitbudget_s']:.0f} s je Fall   ≤ {s['speicher_gb']:.0f} GB   "
          f"Lizenz: {', '.join(s['lizenzen']) if s['lizenzen'] else 'beliebig'}")
    print()

In [ ]:
# Vorgegebene Hilfsfunktion
def empfehle_fuer(situation, zeilen):
    """Filtert die Kandidaten an den harten Anforderungen und sortiert, was übrig bleibt."""
    geprueft = []
    for zeile in zeilen:
        gruende = []
        if zeile["qualitaet"] < situation["mindestqualitaet"]:
            gruende.append(f"Qualität {zeile['qualitaet']:.0%} unter "
                           f"{situation['mindestqualitaet']:.0%}")
        if zeile["sekunden_je_fall"] > situation["zeitbudget_s"]:
            gruende.append(f"{zeile['sekunden_je_fall']:.2f} s über "
                           f"{situation['zeitbudget_s']:.0f} s")
        if zeile["speicher_gb"] > situation["speicher_gb"]:
            gruende.append(f"{zeile['speicher_gb']:.1f} GB über "
                           f"{situation['speicher_gb']:.0f} GB")
        if situation["lizenzen"] and zeile["lizenz"] not in situation["lizenzen"]:
            gruende.append(f"Lizenz {zeile['lizenz']} nicht zugelassen")

        geprueft.append({**zeile, "ausgeschieden": "; ".join(gruende) or None})

    # Erst nach Qualität, bei Gleichstand nach Zeit. Das Minuszeichen dreht die
    # Qualität um, damit beide Schlüssel aufsteigend sortiert werden können.
    uebrig = sorted((z for z in geprueft if z["ausgeschieden"] is None),
                    key=lambda z: (-z["qualitaet"], z["sekunden_je_fall"]))

    return {
        "situation": situation["titel"],
        "empfehlung": uebrig[0]["kandidat"] if uebrig else None,
        "zeilen": geprueft,
    }

## Fazit

Du kannst Kandidaten mit derselben eigenen Eval vergleichen und eine nachvollziehbare Auswahlregel anwenden. Merksatz: **Das kleinste Modell, das alle harten Anforderungen erfüllt, ist oft die beste Wahl.**
